In [1]:
%pip install pymysql
%pip install sqlalchemy

  Using cached pymysql-1.2.0-py3-none-any.whl.metadata (4.3 kB)
Using cached pymysql-1.2.0-py3-none-any.whl (45 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

connection_url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password="XXXXXXXXX",
    host="localhost",
    port=3306,
    database="new_schema"
)

engine = create_engine(connection_url)

df = pd.read_sql(
    "SELECT * FROM superstore_staging_0",
    engine
)

print(df.head())
print(df.shape)

   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156  2016-11-08  2016-11-11    Second Class    CG-12520   
1       2  CA-2016-152156  2016-11-08  2016-11-11    Second Class    CG-12520   
2       3  CA-2016-138688  2016-06-12  2016-06-16    Second Class    DV-13045   
3       4  US-2015-108966  2015-10-11  2015-10-18  Standard Class    SO-20335   
4       5  US-2015-108966  2015-10-11  2015-10-18  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  Postal Code  Region       Product ID         Category Sub-Category  \
0       42420   Sout

In [31]:
df = df.drop(columns=['Row ID','Order ID','Customer ID','Customer Name','City','Product Name','Product ID','Country'])


In [32]:

high_threshold = df.loc[df["Profit"] > 0, "Profit"].median()

print(high_threshold)

13.452


In [33]:
def profit_class(profit):
    if profit < 0:
        return 0
    elif profit < high_threshold:
        return 1
    else:
        return 2

df["Profit Class"] = df["Profit"].apply(profit_class)

In [34]:
df = df.drop(columns=['Postal Code'])

In [35]:
df["Order Date"] = pd.to_datetime(df["Order Date"])


# Month when customer ordered
df["Order Month"] = df["Order Date"].dt.month



# Remove original dates
df.drop(columns=["Order Date", "Ship Date"], inplace=True)

In [36]:

label = df['Profit Class']
df

,Ship Mode,Segment,State,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit,Profit Class,Order Month
0,Second Class,Consumer,Kentucky,South,Furniture,Bookcases,261.9600,2,0.00,41.9136,2,11
1,Second Class,Consumer,Kentucky,South,Furniture,Chairs,731.9400,3,0.00,219.5820,2,11
2,Second Class,Corporate,California,West,Office Supplies,Labels,14.6200,2,0.00,6.8714,1,6
3,Standard Class,Consumer,Florida,South,Furniture,Tables,957.5775,5,0.45,-383.0310,0,10
4,Standard Class,Consumer,Florida,South,Office Supplies,Storage,22.3680,2,0.20,2.5164,1,10
...,...,...,...,...,...,...,...,...,...,...,...,...
19383,Second Class,Consumer,Florida,South,Furniture,Furnishings,25.2480,3,0.20,4.1028,1,1
19384,Standard Class,Consumer,California,West,Furniture,Furnishings,91.9600,2,0.00,15.6332,2,2
19385,Standard Class,Consumer,California,West,Technology,Phones,258.5760,2,0.20,19.3932,2,2
19386,Standard Class,Consumer,California,West,Office Supplies,Paper,29.6000,4,0.00,13.3200,1,2


In [37]:
df.drop(columns=["Profit Class", "Profit"], inplace=True)

In [38]:
df

,Ship Mode,Segment,State,Region,Category,Sub-Category,Sales,Quantity,Discount,Order Month
0,Second Class,Consumer,Kentucky,South,Furniture,Bookcases,261.9600,2,0.00,11
1,Second Class,Consumer,Kentucky,South,Furniture,Chairs,731.9400,3,0.00,11
2,Second Class,Corporate,California,West,Office Supplies,Labels,14.6200,2,0.00,6
3,Standard Class,Consumer,Florida,South,Furniture,Tables,957.5775,5,0.45,10
4,Standard Class,Consumer,Florida,South,Office Supplies,Storage,22.3680,2,0.20,10
...,...,...,...,...,...,...,...,...,...,...
19383,Second Class,Consumer,Florida,South,Furniture,Furnishings,25.2480,3,0.20,1
19384,Standard Class,Consumer,California,West,Furniture,Furnishings,91.9600,2,0.00,2
19385,Standard Class,Consumer,California,West,Technology,Phones,258.5760,2,0.20,2
19386,Standard Class,Consumer,California,West,Office Supplies,Paper,29.6000,4,0.00,2


In [39]:
categorical_cols = [
    "Ship Mode",
    "Segment",
    "State",
    "Region",
    "Category",
    "Sub-Category",
    "Order Month"
]

df_0 = pd.get_dummies(
    df,
    columns=categorical_cols,
    dtype=int
)




categories = {}

for col in categorical_cols:
    categories[col] = df[col].dropna().unique().tolist()



import json

with open("categories.json", "w") as f:
    json.dump(categories, f, indent=4)

In [52]:
import json

with open("feature_columns.json", "w") as f:
    json.dump(df_0.columns.tolist(), f, indent=4)

In [40]:
import torch 

In [41]:
X_tensor = torch.tensor(df_0.values, dtype=torch.float32)
y_tensor = torch.tensor(label.values, dtype=torch.long)


In [ ]:
X_tensor.size()


tensor([2, 2, 1,  ..., 2, 1, 2])

In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tensor,
    y_tensor,
    test_size=0.2,
    random_state=42,
    stratify=y_tensor
)

In [44]:
X_train.size()

torch.Size([15510, 95])

In [45]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# -------------------------
# DataLoaders
# -------------------------
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1024,
    shuffle=False
)

# -------------------------
# Model
# -------------------------
model = nn.Sequential(
    nn.Linear(95, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(0.25),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(0.20),

    nn.Linear(64, 32),
    nn.ReLU(),

    nn.Linear(32, 3)
)

# -------------------------
# Loss and optimizer
# -------------------------
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

# Reduce LR if validation loss stops improving
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5
)

# -------------------------
# Early stopping settings
# -------------------------
epochs = 5000

patience = 200
patience_counter = 0

best_val_loss = float("inf")

# -------------------------
# Training
# -------------------------
for epoch in range(epochs):

    # ===== TRAIN =====
    model.train()

    train_loss = 0

    for X_batch, y_batch in train_loader:

        outputs = model(X_batch)

        loss = loss_fn(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ===== VALIDATION =====
    model.eval()

    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            outputs = model(X_batch)

            loss = loss_fn(outputs, y_batch)

            val_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)

    val_loss /= len(val_loader)

    val_accuracy = correct / total

    # Update learning rate
    scheduler.step(val_loss)

    # -------------------------
    # Early stopping
    # -------------------------
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        patience_counter = 0

        # Save best model
        torch.save(model.state_dict(), "best_model.pth")

    else:
        patience_counter += 1

    if (epoch + 1) % 5 == 0:
        print(
            f"Epoch {epoch+1:3d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy * 100:.2f}% | "
            f"Patience: {patience_counter}/{patience}"
        )

    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch + 1}")
        break


# -------------------------
# Load best model
# -------------------------
model.load_state_dict(torch.load("best_model.pth"))

print("\nBest model loaded.")

Epoch   5 | Train Loss: 0.6908 | Val Loss: 0.7304 | Val Acc: 69.80% | Patience: 0/200
Epoch  10 | Train Loss: 0.5028 | Val Loss: 0.4675 | Val Acc: 84.37% | Patience: 0/200
Epoch  15 | Train Loss: 0.4208 | Val Loss: 0.4053 | Val Acc: 85.17% | Patience: 0/200
Epoch  20 | Train Loss: 0.4049 | Val Loss: 0.3984 | Val Acc: 84.04% | Patience: 4/200
Epoch  25 | Train Loss: 0.3739 | Val Loss: 0.3551 | Val Acc: 85.74% | Patience: 3/200
Epoch  30 | Train Loss: 0.3536 | Val Loss: 0.3477 | Val Acc: 86.28% | Patience: 0/200
Epoch  35 | Train Loss: 0.3365 | Val Loss: 0.3653 | Val Acc: 84.04% | Patience: 1/200
Epoch  40 | Train Loss: 0.3201 | Val Loss: 0.3314 | Val Acc: 86.62% | Patience: 6/200
Epoch  45 | Train Loss: 0.3154 | Val Loss: 0.3602 | Val Acc: 83.65% | Patience: 11/200
Epoch  50 | Train Loss: 0.3129 | Val Loss: 0.3574 | Val Acc: 84.37% | Patience: 4/200
Epoch  55 | Train Loss: 0.3179 | Val Loss: 0.3365 | Val Acc: 85.07% | Patience: 4/200
Epoch  60 | Train Loss: 0.2991 | Val Loss: 0.3967 | V

In [46]:
with torch.no_grad():

    outputs = model(X_test)

    predictions = outputs.argmax(dim=1)

    accuracy = (predictions == y_test).float().mean()

    print("Test Accuracy:", accuracy.item())

Test Accuracy: 0.884218692779541


In [47]:
discount_index = df_0.columns.get_loc("Discount")

print(discount_index)

2


In [48]:
import torch
import numpy as np

def get_discount_ranges(
    model,
    sample,
    discount_index,
    min_discount=0.0,
    max_discount=0.8,
    step=0.001
):
    model.eval()

    class_names = {
        0: "Loss",
        1: "Medium Profit",
        2: "High Profit"
    }

    predictions = []

    with torch.no_grad():
        discount = min_discount

        while discount <= max_discount + 1e-9:
            x = sample.clone()
            x[discount_index] = discount

            output = model(x.unsqueeze(0))
            predicted_class = output.argmax(dim=1).item()

            predictions.append(
                (round(discount, 3), predicted_class)
            )

            discount += step

    # Turn predictions into ranges
    ranges = []

    start_discount = predictions[0][0]
    current_class = predictions[0][1]

    for i in range(1, len(predictions)):
        discount, predicted_class = predictions[i]

        # Class changed -> close previous range
        if predicted_class != current_class:
            end_discount = predictions[i - 1][0]

            ranges.append({
                "start": start_discount,
                "end": end_discount,
                "class": current_class
            })

            start_discount = discount
            current_class = predicted_class

    # Add final range
    ranges.append({
        "start": start_discount,
        "end": predictions[-1][0],
        "class": current_class
    })

    return ranges, class_names

In [49]:
sample = X_test[1]

discount_index = df_0.columns.get_loc("Discount")

ranges, class_names = get_discount_ranges(
    model,
    sample,
    discount_index,
    min_discount=0,
    max_discount=0.8,
    step=0.001
)

In [50]:
print("Discount boundaries:\n")

for r in ranges:
    print(
        f"{r['start'] * 100:.1f}% - "
        f"{r['end'] * 100:.1f}%"
        f"  →  {class_names[r['class']]}"
    )

Discount boundaries:

0.0% - 16.5%  →  Medium Profit
16.6% - 80.0%  →  Loss


At discount 0.0% → Medium Profit (confidence 94.9%)
